# 10 · Análisis y composición de carteras (clase guiada)

**Duración estimada:** 2h 30min (parte teórica)
**Después:** práctica de 1h 30min en `11_carteras_problema.ipynb`

**Lo que vamos a ver hoy:**

1. Qué es una **cartera de inversión** y por qué se diversifica.
2. Cómo descargar precios de acciones con `yfinance`.
3. Cómo pasar de **precios a rentabilidades**.
4. **Retorno esperado, riesgo (volatilidad) y ratio de Sharpe** de una acción.
5. Cómo **componer una cartera** con pesos a medida y ver cómo cambian las métricas.

---

## 1 · ¿Qué es una cartera de inversión?

Una **cartera** es un conjunto de activos (acciones, bonos, etc.) en los que invertimos. A cada activo le asignamos un **peso** (qué porcentaje del dinero total va a cada uno).

**Conceptos clave:**

- **Pesos** `w₁, w₂, …, w_K` → cuánto invertimos en cada activo. Suman 1 (el 100% del capital).
- **Rentabilidad esperada** → cuánto esperamos ganar (de media) en un periodo.
- **Riesgo (volatilidad)** → cómo de "agitada" es la rentabilidad (medida con la desviación típica).
- **Diversificación** → repartir entre varios activos para que el conjunto sea menos volátil que cada activo por separado.

**Idea intuitiva:** si pones todo en una acción y esa acción cae, lo pierdes todo. Si repartes entre 5, el conjunto se mueve menos — aunque tampoco sube tanto si una sola acción sube mucho.

### Mini ejercicio 1 · Piénsalo

Tienes 10 000 € para invertir. Una amiga te ofrece:

- **Opción A:** comprar 1 000 € de cada una de 10 empresas tecnológicas distintas.
- **Opción B:** comprar 10 000 € de una sola empresa de tu sector favorito.

¿Cuál elegirías y por qué? No hay código aquí, es solo para pensar. Apunta tu respuesta en la celda de abajo como comentario.

In [ ]:
#

---

## 2 · Descargamos los datos

Usaremos 5 acciones españolas del IBEX 35 (las mismas que viste en notebooks anteriores) para tener datos familiares.

In [ ]:
# !pip install yfinance -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

### Paso 1 · Descarga los precios de cierre ajustados

In [ ]:
tickers = ['IBE.MC', 'ITX.MC', 'SAN.MC', 'BBVA.MC', 'TEF.MC']
nombres = ['Iberdrola', 'Inditex', 'Santander', 'BBVA', 'Telefónica']
start = '2021-01-01'
end = '2024-12-31'

# auto_adjust=True aplica dividendos y splits
data = yf.download(tickers, start=start, end=end, auto_adjust=True)['Close']
data.columns = nombres
data.head()

### Paso 2 · Echa un vistazo a los datos

In [ ]:
# Tamaño y nulos
print('Shape :', data.shape)
print('\nNulos por columna:')
print(data.isna().sum())

### Paso 3 · Visualiza la evolución (precios normalizados)
Dividimos cada precio por el primero de la serie, así todos empiezan en 1 y son comparables visualmente.

In [ ]:
norm = data / data.iloc[0]
norm.plot(figsize=(11, 5), linewidth=1.5)
plt.ylabel('Precio normalizado (1 = inicio)')
plt.title('Evolución de las 5 acciones (base 1)')
plt.grid(alpha=0.3)
plt.show()

### Mini ejercicio 2 · Observa el gráfico

Mira el gráfico y responde (en la celda de abajo, como comentarios):

1. ¿Qué acción ha sido la más rentable del periodo?
2. ¿Cuál ha sido la más volátil (más "agitada")?
3. Si hubieras invertido 1 000 € en cada una al principio, ¿cuánto tendrías ahora con cada una?

In [ ]:
# # Tu código aquí

---

## 3 · De precios a rentabilidades

En finanzas no trabajamos con precios sino con **rentabilidades**: cuánto ha subido (o bajado) el precio de un día para otro.

**Rentabilidad simple diaria:**

$$r_t = \frac{P_t - P_{t-1}}{P_{t-1}}$$

**Por qué rentabilidades y no precios:** las rentabilidades se pueden sumar (la rentabilidad de 2 días es la suma de las 2 diarias), son aproximadamente simétricas y son comparables entre acciones de precios muy distintos.

### Paso 4 · Calcula la rentabilidad diaria

In [ ]:
retornos = data.pct_change().dropna()
retornos.head()

### Paso 5 · Visualiza la distribución de rentabilidades de Iberdrola

In [ ]:
retornos['Iberdrola'].hist(bins=50, edgecolor='k')
plt.xlabel('Rentabilidad diaria')
plt.ylabel('Frecuencia')
plt.title('Distribución de rentabilidades diarias — Iberdrola')
plt.axvline(0, color='red', linestyle='--')
plt.show()

 La distribución está centrada cerca de 0, es aproximadamente simétrica pero tiene **colas** a izquierda y derecha (subidas y caídas fuertes). Esos valores extremos son los que vimos en la sesión de outliers.

### Mini ejercicio 3 · Compara dos distribuciones

Haz un histograma conjunto (en la misma figura, con `plt.hist` o un gráfico de seaborn) que compare las rentabilidades de Iberdrola e Inditex. ¿Cuál tiene más dispersión? ¿Cuál tiene colas más largas?

In [ ]:
# # Tu código aquí

---

## 4 · Una sola acción: retorno y riesgo

Para una acción, el **retorno medio** y la **volatilidad** se calculan con la media y la desviación típica de sus rentabilidades diarias.

**Anualización:** los mercados abren ~252 días al año. Para comparar acciones en la misma escala temporal, anualizamos:

- Retorno anual ≈ retorno diario medio × 252
- Volatilidad anual ≈ volatilidad diaria × √252

### Paso 6 · Calcula el retorno anualizado y la volatilidad anualizada de cada acción

In [ ]:
resumen = pd.DataFrame({
'retorno_anual_%': (retornos.mean() * 252 * 100).round(2),
'volatilidad_anual_%':(retornos.std() * np.sqrt(252) * 100).round(2),
})
resumen

### Mini ejercicio 4 · Interpreta los resultados

Lee el DataFrame `resumen` y responde (como comentarios):

1. ¿Qué acción ha dado más retorno anual?
2. ¿Cuál ha sido más volátil?
3. ¿La de más retorno es la de más riesgo? (Spoiler: no siempre.)

In [ ]:
# # Tu código aquí

---

## 5 · Componemos una cartera

Una **cartera** es un vector de pesos `w` (uno por acción). Por ejemplo, con 5 acciones, `w = [0.2, 0.2, 0.2, 0.2, 0.2]` es una cartera **equitativa** (mismo peso para todas). `w.sum()` debe ser 1.

**Métricas de la cartera:**

- **Retorno esperado** = `w · μ` (producto escalar del vector de pesos por el vector de rentabilidades medias).
- **Volatilidad** = `√(w · Σ · w)` donde `Σ` es la **matriz de covarianzas** de las rentabilidades.
- **Ratio de Sharpe** = retorno / volatilidad (mayor = mejor relación riesgo/beneficio).

### Paso 7 · Define los pesos y calcula retorno, volatilidad y Sharpe de una cartera equitativa

In [ ]:
# Pesos equitativos
w = np.array([0.2, 0.2, 0.2, 0.2, 0.2])
print('Suma de pesos:', w.sum()) # debe ser 1

In [ ]:
# Vector de rentabilidades medias diarias
mu = retornos.mean()
print(mu)

In [ ]:
# Retorno de la cartera: producto escalar
retorno_cartera = w @ mu
print(f'Retorno diario : {retorno_cartera:.6f}')
print(f'Retorno anual : {retorno_cartera * 252:.4f}')

In [ ]:
# Matriz de covarianzas y volatilidad
cov = retornos.cov()
volatilidad_cartera = np.sqrt(w @ cov @ w)
print(f'Volatilidad diaria: {volatilidad_cartera:.6f}')
print(f'Volatilidad anual : {volatilidad_cartera * np.sqrt(252):.4f}')

In [ ]:
# Ratio de Sharpe
sharpe = (retorno_cartera * 252) / (volatilidad_cartera * np.sqrt(252))
print(f'Sharpe: {sharpe:.3f}')

---

## 6 · Probamos otros pesos manualmente

Cambiando los pesos cambian las métricas. Vamos a ver dos casos extremos para entender la diferencia entre **diversificar** y **concentrar**.

### Paso 8 · Compara la cartera equitativa con una concentrada en Iberdrola

In [ ]:
# Cartera A: equitativa
w_eq = np.array([0.2, 0.2, 0.2, 0.2, 0.2])

# Cartera B: todo en Iberdrola
w_ibe = np.array([1.0, 0.0, 0.0, 0.0, 0.0])

for nombre, w in [('equitativa', w_eq), ('todo Iberdrola', w_ibe)]:
    ret = w @ mu
    vol = np.sqrt(w @ cov @ w)
    s   = (ret * 252) / (vol * np.sqrt(252))
    print(f'{nombre:18s} → retorno {ret*252*100:6.2f}% vol {vol*np.sqrt(252)*100:6.2f}% Sharpe {s:.3f}')

### Mini ejercicio 5 · Crea tu propia cartera

Crea una cartera con pesos a tu gusto (por ejemplo, 50% Inditex y el resto repartido) y compara con las dos anteriores. ¿Tu cartera tiene más o menos Sharpe que la equitativa?

In [ ]:
# # Tu código aquí

---

## Cierre

**Lo que has hecho hoy:**

- Visto qué es una cartera y por qué se diversifica.
- Descargado datos con `yfinance` y visualizado la evolución de las acciones.
- Calculado rentabilidades, retorno esperado, volatilidad y ratio de Sharpe.
- Comprobado que una cartera concentrada en una sola acción se mueve más que la equitativa.

**Lo que NO hemos visto y queda para el problema de mañana:**

- Generar **miles de carteras aleatorias** y buscar la mejor (RETO del notebook 11).
- Visualizar la **frontera eficiente**.
- Optimización matemática con `scipy`.